In [1]:
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

c:\msys64\clang64\lib\python3.12\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
qd.start_client('192.168.0.113')

QICK library version mismatch: 0.2.324 remote (the board), 0.2.302 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


In [4]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = True
default_config.high_threshold = 2000
default_config.low_threshold = 500


default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 5000

default_config.laser_gate_pmod = 0

default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom

# CPMGXY8 with coarse resolution to check

In [6]:
from qickdawg.arqick.arqick_cpmg_XY8 import CPMGXY8

soc = qd.soc
config = copy(default_config)
config.mw_gain = 30000
config.mw_pi2_tns = 50
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)
config.n_cpmg = 1 # number of cpmg xy8 rounds
config.pulse_seq_delay_tus = 1
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = CPMGXY8(config)
prog.run_rounds(soc, rounds=1)


Requested 10 to 100 by 50
Instead using 9.765625 to 58.59375 by 48.828125 in 2 steps


100%|██████████| 2/2 [00:00<00:00, 1330.89it/s]


# 200ps resolution now

In [85]:
'''
RFTest CPMG-XY
=======================================================================
RFTest Envelope class used to test the shape of RF envelopes.
'''

from qickdawg.nvpulsing.nvaverageprogram import NVAveragerProgram
import numpy as np

class RFTest_CPMG(NVAveragerProgram):
    '''
    An NVAveragerProgram class that generates RF gain and frequency stepping sequences.
    '''
    required_cfg = [
        "mw_freg", # Microwave freq # ~1405 MHz per Tommy
        "trigger_gate_pmod", # PMOD pin for external trigger
        
        # Pulse Parameters
        "pi2_len_samples", # length of pi pulse
        "tau_len_samples", # length of tau delay
        "n_cpmg", # number of pulses

        "mw_channel", # MW Channel
        "mw_nqz", # 1 at 1405 MHz
        "mw_gain", #MW Gain
        "reps",

        # Temporary parameters for development
        "trigger_width_treg"]
    
    def initialize(self):
        # NVConfiguration class does not have Gain units unlike freq, time, or phase
        self.check_cfg()

        if self.cfg.mw_gain < 0:
            assert 0, 'Smallest Microwave gain must be postive'
        elif self.cfg.mw_gain > 32767: # 30000 in lockinodmr
            assert 0, 'Largest Microwave gain exceeds maximum value'

        # Get mw registers
        self.declare_gen(ch=self.cfg.mw_channel, nqz=self.cfg.mw_nqz)

        # Configure the waveforms for different sample offsets
        # Waveforms must have at least a length of 3 treg
        self.pi_len_samples = self.cfg.pi2_len_samples * 2
        self.pi_waveform_len_treg = max(int(np.ceil((self.pi_len_samples + 15) / 16)), 3)
        self.half_pi_waveform_len_treg = max(int(np.ceil((self.cfg.pi2_len_samples + 15) / 16)), 3)

        for i in range(16):
            # half pi
            i_data = np.zeros(self.half_pi_waveform_len_treg * 16)
            q_data = np.zeros(self.half_pi_waveform_len_treg * 16)
            i_data[i : i + self.cfg.pi2_len_samples] = 1
            q_data[i : i + self.cfg.pi2_len_samples] = 1
            i_data *= self.soccfg.get_maxv(self.cfg.mw_channel)
            q_data *= self.soccfg.get_maxv(self.cfg.mw_channel)
            self.add_envelope(ch=self.cfg.mw_channel, name=f"half_pi_{i}", idata=i_data, qdata=q_data)

            # pi pulse
            i_data = np.zeros(self.pi_waveform_len_treg * 16)
            q_data = np.zeros(self.pi_waveform_len_treg * 16)
            i_data[i: i + self.pi_len_samples] = 1
            q_data[i: i + self.pi_len_samples] = 1
            i_data *= self.soccfg.get_maxv(self.cfg.mw_channel)
            q_data *= self.soccfg.get_maxv(self.cfg.mw_channel)
            self.add_envelope(ch=self.cfg.mw_channel, name=f"pi_{i}", idata=i_data, qdata=q_data)

        # Compute how much delay is in the waveforms
        self.pi_len_unused = self.pi_waveform_len_treg*16 - self.pi_len_samples
        self.half_pi_len_unused = self.half_pi_waveform_len_treg*16 - self.cfg.pi2_len_samples

        # Set up registers for storing tau treg and sample offsets
        self.tau_samples = self.new_gen_reg(self.cfg.mw_channel,
                                            name='tau_step',
                                            init_val=self.cfg.tau_len_samples - self.pi_len_unused)
        
        # we can initialize sample_offset to already account for the first half_pi_pulse
        self.sample_offset = self.new_gen_reg(self.cfg.mw_channel,
                                                    name='sample_offset',
                                                    init_val=(self.pi_len_unused-self.half_pi_len_unused))
        self.treg_offset = self.new_gen_reg(self.cfg.mw_channel,
                                                name='treg_offset',
                                                init_val=0)
        
        # CPMG loop register
        self.n_cpmg_register = self.new_gen_reg(self.cfg.mw_channel,
                                                    name='ncpmg',
                                                    init_val=self.cfg.n_cpmg - 1)
        
        # comparison register
        self.comparison = self.new_gen_reg(self.cfg.mw_channel,
                                                    name='comparison_val',
                                                    init_val=0)
        
        self.default_pulse_registers(ch=self.cfg.mw_channel,
                                     style='arb',
                                     freq=self.cfg.mw_freg,
                                     gain=self.cfg.mw_gain)
        
        # Set first half pi x
        self.set_pulse_registers(ch=self.cfg.mw_channel, waveform="half_pi_0", phase=0)

        # CPMG waveform
        self.synci(200)  # give processor some time to configure pulses


    def body(self):
        # Half pi pulse, 0 sample offset, phase = x
        self.offset_computations() # offset comp is for the very next sync but the one after pulse
        self.pulse(ch=self.cfg.mw_channel)
        self.sync_all()

        self.sync(self.treg_offset.page, self.treg_offset.addr)

        # Loop pi-X, tau, pi-Y, tau
        self.n_cpmg_register.reset()
        self.label("LOOP_ncpmg")
        
        # X pulse
        # Configures assembly code for picking the waveform
        self.set_waveform("Execute_X_Pi_Pulse", "pi_", phase=0)
        self.offset_computations()
        self.pulse(ch=self.cfg.mw_channel)
        self.sync_all()

        self.sync(self.treg_offset.page, self.treg_offset.addr)

        # Y pulse
        self.set_waveform("Execute_Y_Pi_Pulse", "pi_", phase=90)
        self.offset_computations()
        self.pulse(ch=self.cfg.mw_channel)
        self.sync_all()
        self.sync(self.treg_offset.page, self.treg_offset.addr)

        self.loopnz(
                self.n_cpmg_register.page,
                self.n_cpmg_register.addr,
                'LOOP_ncpmg')
        
        # Pi/2 X pulse
        self.set_waveform("Execute_Last_Pulse", "half_pi_", phase=0)
        self.pulse(ch=self.cfg.mw_channel)
        self.sync_all()

        # Reset for next loop
        self.set_pulse_registers(ch=self.cfg.mw_channel, waveform="half_pi_0", phase=0)
        self.sample_offset.reset() # reset the sample_offset adjustment 

  
    def set_waveform(self, label, pulse_type="pi_", phase=0):
        """
        Configures the assembly code necessary for setting the waveform
        """
        self.select_waveform(4, 8, 16, label, pulse_type, phase)
        self.label(label)
    
    def offset_computations(self):
        """
        Compute the sample_offset and treg_offset for the next pulse
        """
        # sample_offset = sample_offset + sample_step
        self.math(self.sample_offset.page, self.sample_offset.addr, self.sample_offset.addr, "+", self.tau_samples.addr)
        # treg_offset = sample_offset >> 4 (global offset)
        self.mathi(self.sample_offset.page, self.treg_offset.addr, self.sample_offset.addr, ">>", 4)
        # sample_offset = sample_offset & 15
        self.mathi(self.sample_offset.page, self.sample_offset.addr, self.sample_offset.addr, "&", 15)

    def select_waveform(self, depth, center, span, label, pulse_type="pi_", phase=0):
        """
        A binary search tree to select the correct waveform for a given sample_offset
        """
        center = int(center)
        if (depth==0):
            self.set_pulse_registers(ch=self.cfg.mw_channel, waveform=f"{pulse_type}{center}", phase=self.deg2reg(phase))
            self.condj(self.sample_offset.page, self.sample_offset.addr, "==", self.sample_offset.addr, label)
            return

        self.regwi(self.comparison.page, self.comparison.addr, center)
        self.condj(
                self.sample_offset.page,
                self.sample_offset.addr,
                ">=",
                self.comparison.addr,
                f"{pulse_type}{phase}_pulse_offset_{center}")
        
        self.select_waveform(depth-1, center-span/4, span/2, label, pulse_type, phase)

        self.label(f"{pulse_type}{phase}_pulse_offset_{center}")
        self.select_waveform(depth-1, center+span/4, span/2, label, pulse_type, phase)

In [ ]:
import copy
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.mw_fMHz = 500 # 200
config.mw_gain = 32000
config.mw_nqz = 1
# Timing params
config.pi2_len_samples = 200
config.tau_len_samples = 100
# Sweep params
config.n_cpmg = 2
config.reps = 1
# Triggering
config.trigger_width_tns = 50
config.trigger_gate_pmod = 0
#config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)


prog = RFTest_CPMG(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 1/1 [00:00<00:00, 999.60it/s]
